In [ ]:
"""
exp02 Step 03: Walk-Forward Evaluation (PROTOCOL.md §8-§12)

Per fold (3-month windows, 20-session embargo):
    1. VALIDATION: fit the re-entry model (logistic regression; LightGBM as a diagnostic) on each training window
       (10 years, all history), then simulate every candidate (k x threshold offset) on the validation window.
    2. SELECTION: mean validation excess return over the last 4 validation quarters (pooled), per candidate.
    3. HONEST REPORTING (every run mode, validation data only):
         - every candidate chained over the validation quarters (after the fact: optimistic for the best one);
         - the PRIOR-ONLY path: the candidate selected at fold f, re-simulated on fold f + 1's validation quarter with its
           baselines (same stop + fixed-delay / random re-entry, buy-and-hold, 200-session trend rule): the honest
           validation estimate and the information test;
         - time in cash, the minimum detectable effect, the success criteria and the monthly bootstrap on that path.
    4. TEST (run modes "latest" / "history" only): refit, simulate the test window ONCE, with the same baselines.

Success (PROTOCOL.md §3): primary = chained total return above buy-and-hold; secondary = higher Sharpe AND smaller
maximum drawdown, counted only when the return is at least 90% of buy-and-hold's. The model adds value only if it also
beats the same-stop baselines. STOP (§11.3) if the prior-only path fails to beat buy-and-hold and the fixed-delay baselines.

RUN_MODE:
    "validation_only"   every fold WITHOUT its test window (use until the design is frozen)
    "latest"            the most recent test window (needs the validation of the last 4 folds)
    "history"           every test window (historical replay)
Prerequisite: the raw data only. Outputs: store04_experiments/exp02_stop_reentry/step03_walk_forward_data/.
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
import time
# IMPORT PLOTLY
import plotly.express as px
import plotly.graph_objects as go
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp02_stop_reentry import config as exp_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp02_stop_reentry.walk_forward import get_fold_pdf, run_validation_candidate_pdf, get_pooled_selection_pdf, run_window_with_baseline_dict
# IMPORT THE EVALUATION FUNCTIONS
from so.core.evaluation import get_prior_only_pdf, get_candidate_summary_pdf, get_window_return_summary_dict, get_baseline_summary_pdf, \
                               get_chained_daily_return_pdf, get_excess_return_bootstrap_dict, get_success_criteria_dict
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [ ]:
# DEFINE THE RUN MODE ("validation_only", "latest" or "history"): NEVER "latest" / "history" BEFORE THE DESIGN IS FROZEN
RUN_MODE_STR = "validation_only"
# DEFINE WHETHER THE TEST WINDOWS ARE EVALUATED
RUN_TEST_BOOL = RUN_MODE_STR in ["latest", "history"]
# DISPLAY THE SETTINGS
print(f"Run Mode:\t{RUN_MODE_STR}\tTest Windows Evaluated:\t{RUN_TEST_BOOL}")
print(f"Training Windows:\t{exp_config.TRAIN_WINDOW_LIST}\tStop k:\t{exp_config.STOP_K_LIST}\tThreshold Offsets:\t{exp_config.REENTRY_THRESHOLD_OFFSET_LIST}")
print(f"Max Cash Sessions:\t{exp_config.MAX_CASH_SESSIONS}\tEmbargo:\t{exp_config.EMBARGO_TRADING_DAYS} sessions\tModel:\t{exp_config.PRIMARY_MODEL_STR} (diagnostic: {exp_config.DIAGNOSTIC_MODEL_STR})")

In [ ]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step03_walk_forward_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "validation")

In [ ]:
# CALL FUNCTION TO READ THE COMPLETE MINUTE BARS (RAW FOLDER OF so.paths)
complete_ohlcv_pdf = get_complete_ohlcv_pdf()
# CALL FUNCTION TO CONVERT THE COMPLETE DATA TO ARRAYS (USED BY THE SIMULATOR)
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
# CALL FUNCTION TO BUILD THE DAILY FEATURES AND TARGET (SECONDS; pipeline/step06 SAVES THE SAME TABLE AS A RECORD)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
# DISPLAY INFORMATION
print(f"min timestamp:\t{complete_ohlcv_pdf.timestamp.min()}")
print(f"max timestamp:\t{complete_ohlcv_pdf.timestamp.max()}")
print(f"Sessions:\t{len(daily_pdf):,}\tLabelled:\t{int(daily_pdf['y_fwd_positive'].notna().sum()):,}")

In [ ]:
# CALL FUNCTION TO GET THE FOLDS
fold_pdf = get_fold_pdf(ohlcv_array_dict["session_date_list"])
# SELECT THE FOLDS WHOSE VALIDATION IS NEEDED ("latest" NEEDS THE LAST 4 FOLDS FOR THE POOLED SELECTION)
validation_fold_pdf = fold_pdf.iloc[-exp_config.SELECTION_POOLED_QUARTER_COUNT:] if RUN_MODE_STR == "latest" else fold_pdf
# SELECT THE FOLDS WHOSE TEST WINDOW IS EVALUATED
test_fold_pdf = fold_pdf.iloc[[-1]] if RUN_MODE_STR == "latest" else (fold_pdf if RUN_MODE_STR == "history" else fold_pdf.iloc[0:0])
# SAVE THE SCHEDULE
write_csv_file_to_path(fold_pdf, f"{output_path_str}fold_schedule_data.csv", "W", alert_in=False)
# DISPLAY INFORMATION
print(f"Fold Count:\t{len(fold_pdf)}\tValidation Folds:\t{len(validation_fold_pdf)}\tTest Folds:\t{len(test_fold_pdf)}")
# PREVIEW DATAFRAME
fold_pdf[["fold_id", "train_start_10y", "train_start_all", "train_end", "valid_start", "valid_end", "test_start", "test_end"]]

In [ ]:
# LIST TO HOLD THE CANDIDATE TABLES
candidate_pdf_list = []
# ITERATE OVER THE VALIDATION FOLDS
for _, fold_row in validation_fold_pdf.iterrows():
    # DEFINE THE FOLD ID AND START TIME
    fold_id, start_time = int(fold_row["fold_id"]), time.time()
    # CALL FUNCTION TO RUN THE VALIDATION CANDIDATES
    fold_candidate_pdf = run_validation_candidate_pdf(fold_row, daily_pdf, ohlcv_array_dict)
    candidate_pdf_list.append(fold_candidate_pdf)
    # LOG EVERY CANDIDATE AS A TRIAL (ONE WRITE PER FOLD)
    log_trial_pdf(exp_config, "validation", fold_candidate_pdf, ["train_window", "stop_k", "threshold_offset"],
                  [col for col in fold_candidate_pdf.columns if col.startswith("valid_")], False, fold_pdf)
    # SAVE THE CANDIDATES
    write_csv_file_to_path(fold_candidate_pdf, f"{output_path_str}fold{fold_id:03d}_validation_candidate_data.csv", "W", alert_in=False)
    # DISPLAY INFORMATION
    best_row = fold_candidate_pdf.sort_values("valid_excess_return", ascending=False).iloc[0]
    print(f"Fold {fold_id}:\tvalidation {fold_row['valid_start']} -> {fold_row['valid_end']}\tbuy & hold {best_row['valid_buy_hold_total_return']:.2%}"
          f"\tbest candidate this quarter {best_row['valid_total_return']:.2%}\t({(time.time() - start_time):.0f} s)")
# CONCATENATE THE CANDIDATES AND SAVE THEM
candidate_pdf = pd.concat(candidate_pdf_list, ignore_index=True)
write_csv_file_to_path(candidate_pdf, f"{output_path_str}validation_candidate_data.csv", "W")

In [ ]:
"""
After-the-fact view (OPTIMISTIC for the best candidate: it is the best of 18 chosen with hindsight). The minimum detectable
effect (mde_per_year) is the true yearly excess return a 5% test on 44 quarters would detect with 80% power.
"""
# CALL FUNCTION TO SUMMARIZE EVERY CANDIDATE OVER THE VALIDATION QUARTERS
candidate_summary_pdf = get_candidate_summary_pdf(candidate_pdf, ["train_window", "stop_k", "threshold_offset"],
                                                  ["valid_stop_count", "valid_cash_session_pct", "valid_mean_share_gain_pct"])
write_csv_file_to_path(candidate_summary_pdf, f"{output_path_str}candidate_summary_data.csv", "W", alert_in=False)
# DISPLAY THE CLASSIFICATION DIAGNOSTICS (MEAN VALIDATION AUC: 0.5 = NO RANKING ABILITY; SHARE OF FOLDS ABOVE 0.5)
auc_pdf = candidate_pdf.drop_duplicates(["fold_id", "train_window"])
display(auc_pdf.groupby("train_window")[[f"valid_{exp_config.PRIMARY_MODEL_STR}_auc", f"valid_{exp_config.DIAGNOSTIC_MODEL_STR}_auc"]].agg(["mean", lambda s: float((s > 0.5).mean())]).round(3))
# PREVIEW DATAFRAME
candidate_summary_pdf.round(4)

In [ ]:
"""
Prior-only path (the honest validation estimate): the candidate selected at fold f (pooled over the 4 quarters up to f)
is evaluated on fold f + 1's validation quarter, with its baselines. It is computed in "validation_only" mode too:
no test window is read.
"""
# CALL FUNCTION TO SELECT ONE CANDIDATE PER FOLD (POOLED OVER THE LAST 4 VALIDATION QUARTERS)
selection_pdf = get_pooled_selection_pdf(candidate_pdf)
write_csv_file_to_path(selection_pdf, f"{output_path_str}selection_data.csv", "W", alert_in=False)
# CALL FUNCTION TO GET THE PRIOR-ONLY PATH (SELECTION OF FOLD f EVALUATED ON FOLD f + 1)
prior_only_pdf = get_prior_only_pdf(candidate_pdf, selection_pdf, ["train_window", "stop_k", "threshold_offset"])
# LISTS TO HOLD THE BASELINES AND THE DAILY EQUITY OF THE PATH
prior_baseline_pdf_list, prior_equity_pdf_list, prior_buy_hold_equity_pdf_list = [], [], []
# ITERATE OVER THE EVALUATED FOLDS
for _, prior_row in prior_only_pdf.iterrows():
    # CALL FUNCTION TO RE-SIMULATE THE SELECTED CANDIDATE ON THE QUARTER WITH ITS BASELINES
    fold_row = fold_pdf[fold_pdf["fold_id"] == prior_row["fold_id"]].iloc[0]
    window_dict = run_window_with_baseline_dict(fold_row, prior_row, daily_pdf, ohlcv_array_dict, "valid", alert_in=False)
    # CHECK THE RE-SIMULATION EQUALS THE LOGGED CANDIDATE
    assert np.isclose(window_dict["metric_dict"]["total_return"], prior_row["valid_total_return"]), f"fold {prior_row['fold_id']}: re-simulation differs"
    # COLLECT THE RESULTS
    prior_baseline_pdf_list.append(window_dict["baseline_pdf"].assign(fold_id=int(prior_row["fold_id"])))
    prior_equity_pdf_list.append(window_dict["daily_equity_pdf"])
    prior_buy_hold_equity_pdf_list.append(window_dict["buy_hold_daily_equity_pdf"])
# CONCATENATE THE BASELINES AND SAVE THE PATH
prior_baseline_pdf = pd.concat(prior_baseline_pdf_list, ignore_index=True)
write_csv_file_to_path(prior_only_pdf, f"{output_path_str}prior_only_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_baseline_pdf, f"{output_path_str}prior_only_baseline_data.csv", "W", alert_in=False)
# SUMMARIZE THE PATH (QUARTERS), THE BASELINES (INFORMATION TEST) AND THE DAILY CRITERIA
prior_summary_dict = get_window_return_summary_dict(prior_only_pdf["valid_total_return"].to_numpy(), prior_only_pdf["valid_buy_hold_total_return"].to_numpy())
baseline_summary_pdf = get_baseline_summary_pdf(prior_baseline_pdf)
prior_return_pdf, prior_buy_hold_return_pdf = get_chained_daily_return_pdf(prior_equity_pdf_list), get_chained_daily_return_pdf(prior_buy_hold_equity_pdf_list)
baseline_chained_dict = {row["baseline"]: row["chained_return"] for _, row in baseline_summary_pdf.iterrows() if row["baseline"].startswith(("fixed_delay", "random"))}
prior_success_dict = get_success_criteria_dict(prior_return_pdf, prior_buy_hold_return_pdf, baseline_chained_dict)
prior_bootstrap_dict = get_excess_return_bootstrap_dict(prior_return_pdf, prior_buy_hold_return_pdf)
# DISPLAY INFORMATION
print(f"Prior-only path ({prior_summary_dict['window_count']} quarters):\t{prior_summary_dict['chained_return']:.2%} vs buy & hold {prior_summary_dict['chained_buy_hold_return']:.2%}"
      f"\t(beats buy & hold in {prior_summary_dict['windows_beating_buy_hold']} quarters, one-sided sign test p = {prior_summary_dict['sign_test_p_value']:.3f})")
print(f"Mean excess per quarter:\t{prior_summary_dict['mean_excess_return']:.2%} (SD {prior_summary_dict['std_excess_return']:.2%}, t = {prior_summary_dict['t_stat']:.2f})"
      f"\tMinimum detectable effect:\t{prior_summary_dict['mde_per_window']:.2%} per quarter, about {prior_summary_dict['mde_per_year']:.1%} per year")
print(f"Mean time in cash:\t{prior_only_pdf['valid_cash_session_pct'].mean():.1%}")
print(f"PRIMARY on the prior-only path:\t{prior_success_dict['primary_success']}\tComparable return:\t{prior_success_dict['comparable_return']}"
      f"\tSecondary Sharpe / drawdown:\t{prior_success_dict['secondary_sharpe']} / {prior_success_dict['secondary_drawdown']}")
print(f"Excess return 95% interval (monthly bootstrap):\t[{prior_bootstrap_dict['ci_low']:.2%}, {prior_bootstrap_dict['ci_high']:.2%}]\tstatistically supported: {prior_bootstrap_dict['supported']}")
# PREVIEW THE INFORMATION TEST (THE MODEL'S CHAINED RETURN vs EACH BASELINE)
baseline_summary_pdf.round(4)

In [ ]:
# CREATE A FIGURE OF THE PRIOR-ONLY PATH vs BUY AND HOLD AND THE FIXED-DELAY BASELINES (CHAINED QUARTERS)
plot_pdf = prior_baseline_pdf[prior_baseline_pdf["baseline"].isin(["model", "buy_hold", "fixed_delay_1", "fixed_delay_5", "fixed_delay_20", f"trend_ma{exp_config.TREND_RULE_MA_SESSIONS}"])] \
    .sort_values("fold_id").assign(growth=lambda df: df.groupby("baseline")["total_return"].transform(lambda s: np.cumprod(1 + s)))
fig = px.line(plot_pdf, x="fold_id", y="growth", color="baseline", markers=True, template="plotly_dark", width=1100, height=500,
              title="Prior-Only Path On Validation Quarters (growth of 1, chained quarters)")
fig.show()

In [ ]:
# LOG ONE SUMMARY ENTRY OF THE RUN (THE CANDIDATES ARE ALREADY LOGGED; THIS RECORDS THE HONEST ESTIMATE THAT WAS SEEN)
summary_trial_dict = log_trial_dict(exp_config, "summary", {"run_mode": RUN_MODE_STR, "selection": f"pooled {exp_config.SELECTION_POOLED_QUARTER_COUNT} quarters", "path": "prior-only on validation"},
               {**prior_summary_dict, **{f"success_{key}": value for key, value in prior_success_dict.items() if not isinstance(value, dict)},
                "bootstrap_ci_low": prior_bootstrap_dict["ci_low"], "bootstrap_ci_high": prior_bootstrap_dict["ci_high"],
                "baseline_chained_return_dict": baseline_chained_dict, "best_after_the_fact_chained_return": float(candidate_summary_pdf["chained_return"].iloc[0])},
               False, note_str_in="exp02 step03 validation summary (prior-only path, baselines, MDE)")

In [ ]:
# LISTS TO HOLD THE TEST RESULTS
fold_summary_dict_list, strategy_equity_pdf_list, buy_hold_equity_pdf_list, baseline_pdf_list = [], [], [], []
# ITERATE OVER THE TEST FOLDS (NONE IN validation_only MODE)
for _, fold_row in test_fold_pdf.iterrows():
    # COLLECT THE FOLD AND ITS SELECTED CANDIDATE
    fold_id = int(fold_row["fold_id"])
    selection_row = selection_pdf[selection_pdf["fold_id"] == fold_id].iloc[0]
    print(f"\nFold {fold_id}:\ttest {fold_row['test_start']} -> {fold_row['test_end']}\tselected {selection_row['train_window']}, k={selection_row['stop_k']}, offset={selection_row['threshold_offset']}")
    # CALL FUNCTION TO EVALUATE THE TEST WINDOW ONCE
    test_dict = run_window_with_baseline_dict(fold_row, selection_row, daily_pdf, ohlcv_array_dict, "test")
    # LOG THE TEST EVALUATION
    log_trial_dict(exp_config, "test", {"train_window": selection_row["train_window"], "stop_k": selection_row["stop_k"], "threshold_offset": selection_row["threshold_offset"]},
                   test_dict["metric_dict"], True, fold_row)
    # SAVE THE TEST OUTPUTS
    for name_str in ["baseline_pdf", "cost_sensitivity_pdf", "transaction_pdf", "episode_pdf", "daily_equity_pdf"]:
        write_csv_file_to_path(test_dict[name_str], f"{output_path_str}fold{fold_id:03d}_test_{name_str.replace('_pdf', '')}_data.csv", "W", alert_in=False)
    # COLLECT THE RESULTS
    strategy_equity_pdf_list.append(test_dict["daily_equity_pdf"])
    buy_hold_equity_pdf_list.append(test_dict["buy_hold_daily_equity_pdf"])
    baseline_pdf_list.append(test_dict["baseline_pdf"].assign(fold_id=fold_id))
    fold_summary_dict_list.append({"fold_id": fold_id, "test_start": fold_row["test_start"], "test_end": fold_row["test_end"],
                                   "train_window": selection_row["train_window"], "stop_k": selection_row["stop_k"], "threshold_offset": selection_row["threshold_offset"],
                                   **test_dict["metric_dict"]})
# CONVERT THE FOLD SUMMARIES
fold_summary_pdf = pd.DataFrame(fold_summary_dict_list)

In [ ]:
# IF THERE ARE TEST RESULTS
if not fold_summary_pdf.empty:
    # SAVE THE FOLD SUMMARY
    write_csv_file_to_path(fold_summary_pdf, f"{output_path_str}fold_summary_data.csv", "W")
    # CHAIN THE DAILY RETURNS OF THE TEST WINDOWS AND SUMMARIZE THE BASELINES
    strategy_return_pdf, buy_hold_return_pdf = get_chained_daily_return_pdf(strategy_equity_pdf_list), get_chained_daily_return_pdf(buy_hold_equity_pdf_list)
    test_baseline_summary_pdf = get_baseline_summary_pdf(pd.concat(baseline_pdf_list, ignore_index=True))
    test_baseline_chained_dict = {row["baseline"]: row["chained_return"] for _, row in test_baseline_summary_pdf.iterrows() if row["baseline"].startswith(("fixed_delay", "random"))}
    # APPLY THE SUCCESS CRITERIA AND THE BOOTSTRAP
    success_dict = get_success_criteria_dict(strategy_return_pdf, buy_hold_return_pdf, test_baseline_chained_dict)
    bootstrap_dict = get_excess_return_bootstrap_dict(strategy_return_pdf, buy_hold_return_pdf)
    test_summary_dict = get_window_return_summary_dict(fold_summary_pdf["total_return"].to_numpy(), fold_summary_pdf["buy_hold_total_return"].to_numpy())
    # DISPLAY INFORMATION
    print(f"Chained total return:\tmodel {success_dict['strategy_total_return']:.2%} | buy & hold {success_dict['buy_hold_total_return']:.2%}")
    print(f"PRIMARY (return above buy & hold):\t{success_dict['primary_success']}")
    print(f"Excess return 95% interval (monthly bootstrap):\t[{bootstrap_dict['ci_low']:.2%}, {bootstrap_dict['ci_high']:.2%}]\tstatistically supported: {bootstrap_dict['supported']}")
    print(f"Comparable return (>= {success_dict['comparable_floor']:.2%}):\t{success_dict['comparable_return']}")
    print(f"SECONDARY Sharpe:\t{success_dict['strategy_sharpe_ratio']:.2f} vs {success_dict['buy_hold_sharpe_ratio']:.2f}\t-> {success_dict['secondary_sharpe']}")
    print(f"SECONDARY max drawdown:\t{success_dict['strategy_max_drawdown']:.2%} vs {success_dict['buy_hold_max_drawdown']:.2%}\t-> {success_dict['secondary_drawdown']}")
    print(f"Same-stop baselines beaten:\t{success_dict['baselines_beaten']}")
    print(f"Folds beating buy & hold:\t{test_summary_dict['windows_beating_buy_hold']}/{test_summary_dict['window_count']}\t(one-sided sign test p-value: {test_summary_dict['sign_test_p_value']:.4f})")
# PREVIEW DATAFRAME
fold_summary_pdf

In [ ]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)